In [1]:
!python -m pip install -q pandas numpy scikit-learn



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%%writefile code.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterable, Literal, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)

READMITTED_CLASSES_3: Tuple[str, str, str] = ("NO", "<30", ">30")


class TargetEngineeringError(ValueError):
    """Raised when the readmitted target contains unexpected/invalid values."""


def _normalize_readmitted_value(v: object) -> Optional[str]:
    """
    Normalize a single readmitted value:
    - None/NaN -> None
    - strings -> stripped, uppercased (except <30 and >30 remain as-is after upper)
    """
    if v is None:
        return None
    # pandas NA / numpy nan
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass

    s = str(v).strip()
    if s == "":
        return None
    return s.upper()


def validate_readmitted_values(
    values: Union[pd.Series, Iterable[object]],
    allowed: Sequence[str] = READMITTED_CLASSES_3,
    *,
    allow_na: bool = False,
) -> None:
    """
    Validate that all non-missing readmitted values are inside 'allowed'.

    Raises:
        TargetEngineeringError if an unexpected value is found.
    """
    allowed_set = {a.upper() for a in allowed}
    if isinstance(values, pd.Series):
        raw_iter = values.tolist()
    else:
        raw_iter = list(values)

    unexpected = set()
    for v in raw_iter:
        nv = _normalize_readmitted_value(v)
        if nv is None:
            if not allow_na:
                # Missing values are unexpected unless allow_na=True
                unexpected.add(None)
            continue
        if nv not in allowed_set:
            unexpected.add(nv)

    if unexpected:
        raise TargetEngineeringError(
            f"Unexpected readmitted values found: {sorted([x for x in unexpected if x is not None])}"
            + (" (and missing values)" if None in unexpected else "")
            + f". Allowed values: {list(allowed)}."
        )


def binarize_readmitted(
    readmitted: pd.Series,
    *,
    positive_value: str = "<30",
    negative_values: Sequence[str] = ("NO", ">30"),
    output_name: str = "readmitted_30d",
    unknown_policy: Literal["error", "nan"] = "error",
) -> pd.Series:
    """
    Convert {NO, <30, >30} into binary {1 if <30, 0 otherwise}.

    Args:
        readmitted: pandas Series with original readmitted values.
        positive_value: value mapped to 1.
        negative_values: values mapped to 0.
        output_name: name for the output series.
        unknown_policy:
            - "error": raise if any unexpected/NA values appear
            - "nan": map unexpected/NA to <NA> (nullable integer)

    Returns:
        pd.Series of dtype int8 (or nullable Int8 if unknown_policy="nan").
    """
    pos = positive_value.upper()
    neg = tuple(v.upper() for v in negative_values)

    # Normalize series
    norm = readmitted.map(_normalize_readmitted_value)

    mapping: Dict[str, int] = {pos: 1, **{v: 0 for v in neg}}

    if unknown_policy == "error":
        validate_readmitted_values(norm, allowed=(pos, *neg), allow_na=False)
        out = norm.map(mapping).astype(np.int8)
        out.name = output_name
        return out

    if unknown_policy == "nan":
        # allow NA/unknown -> <NA>
        out = norm.map(mapping)
        out = out.astype("Int8")  # nullable integer supports <NA>
        out.name = output_name
        return out

    raise ValueError(f"unknown_policy must be 'error' or 'nan', got: {unknown_policy}")


def keep_multiclass_readmitted(
    readmitted: pd.Series,
    *,
    output_name: str = "readmitted_3class",
    allowed: Sequence[str] = READMITTED_CLASSES_3,
    unknown_policy: Literal["error", "nan"] = "error",
) -> pd.Series:
    """
    Keep the original 3-class label, optionally validating values.

    Returns:
        pd.Series of dtype 'category' with categories in the allowed order,
        or with missing if unknown_policy="nan".
    """
    norm = readmitted.map(_normalize_readmitted_value)

    if unknown_policy == "error":
        validate_readmitted_values(norm, allowed=allowed, allow_na=False)
    elif unknown_policy == "nan":
        # allow NA/unknown; do not raise
        pass
    else:
        raise ValueError(f"unknown_policy must be 'error' or 'nan', got: {unknown_policy}")

    cat = pd.Categorical(norm, categories=[a.upper() for a in allowed], ordered=False)
    out = pd.Series(cat, index=readmitted.index, name=output_name)
    return out


def engineer_targets(
    df: pd.DataFrame,
    *,
    source_col: str = "readmitted",
    binary_col: str = "readmitted_30d",
    multiclass_col: str = "readmitted_3class",
    add_multiclass: bool = True,
    drop_source: bool = False,
    unknown_policy: Literal["error", "nan"] = "error",
) -> pd.DataFrame:
    """
    Add engineered target columns to a dataframe:
      - binary: 1 if <30 else 0
      - optional multiclass: categorical {NO, <30, >30}

    This keeps Section 2 self-contained and reproducible.

    Returns:
        A copy of df with new columns added (and optionally source_col dropped).
    """
    if source_col not in df.columns:
        raise KeyError(f"Column '{source_col}' not found in df. Available: {list(df.columns)}")

    out = df.copy()
    out[binary_col] = binarize_readmitted(
        out[source_col],
        output_name=binary_col,
        unknown_policy=unknown_policy,
    )

    if add_multiclass:
        out[multiclass_col] = keep_multiclass_readmitted(
            out[source_col],
            output_name=multiclass_col,
            unknown_policy=unknown_policy,
        )

    if drop_source:
        out.drop(columns=[source_col], inplace=True)

    return out


@dataclass(frozen=True)
class BinaryConfusionTerms:
    tp: int
    fp: int
    fn: int
    tn: int


def confusion_terms_binary(
    y_true: Union[pd.Series, np.ndarray, Sequence[int]],
    y_pred: Union[pd.Series, np.ndarray, Sequence[int]],
    *,
    positive_label: int = 1,
) -> BinaryConfusionTerms:
    """
    Return TP/FP/FN/TN for a binary task once the positive class is defined.

    This directly supports the Section 2 narrative about TP/FP/FN/TN meaning.
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    # cm layout with labels [0,1] is:
    # [[TN, FP],
    #  [FN, TP]]
    tn, fp, fn, tp = cm.ravel()
    return BinaryConfusionTerms(tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn))


def metrics_binary(
    y_true: Union[pd.Series, np.ndarray, Sequence[int]],
    y_pred: Union[pd.Series, np.ndarray, Sequence[int]],
    y_score: Optional[Union[pd.Series, np.ndarray, Sequence[float]]] = None,
) -> Dict[str, Optional[float]]:
    """
    Convenience metric bundle for binary framing (Section 2 awareness):
      - accuracy
      - f1
      - balanced_accuracy
      - roc_auc (if y_score is provided)
    """
    res: Dict[str, Optional[float]] = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, pos_label=1)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "roc_auc": None,
    }
    if y_score is not None:
        res["roc_auc"] = float(roc_auc_score(y_true, y_score))
    return res


def metrics_multiclass(
    y_true: Union[pd.Series, np.ndarray, Sequence[str]],
    y_pred: Union[pd.Series, np.ndarray, Sequence[str]],
    y_proba: Optional[np.ndarray] = None,
    *,
    labels: Sequence[str] = READMITTED_CLASSES_3,
) -> Dict[str, Optional[float]]:
    """
    Metric bundle for the advanced extension (3-class framing):
      - macro_f1
      - weighted_f1
      - balanced_accuracy (macro recall)
      - ovr_auc_macro (if y_proba is provided with shape [n_samples, n_classes])

    Notes:
      - For AUC, we use one-vs-rest multi-class AUC (macro).
      - Requires y_proba columns correspond to 'labels' in the same order.
    """
    labels_up = [l.upper() for l in labels]

    yt = pd.Series(y_true).map(_normalize_readmitted_value)
    yp = pd.Series(y_pred).map(_normalize_readmitted_value)

    res: Dict[str, Optional[float]] = {
        "macro_f1": float(f1_score(yt, yp, labels=labels_up, average="macro")),
        "weighted_f1": float(f1_score(yt, yp, labels=labels_up, average="weighted")),
        "balanced_accuracy": float(balanced_accuracy_score(yt, yp)),
        "ovr_auc_macro": None,
    }

    if y_proba is not None:
        y_proba = np.asarray(y_proba)
        if y_proba.ndim != 2 or y_proba.shape[1] != len(labels_up):
            raise ValueError(
                f"y_proba must have shape [n_samples, {len(labels_up)}] matching labels order {labels_up}."
            )
        res["ovr_auc_macro"] = float(
            roc_auc_score(yt, y_proba, multi_class="ovr", average="macro", labels=labels_up)
        )

    return res


Writing code.py


In [1]:
%%writefile test_target_engineering.py
import os
import unittest
import importlib.util

import numpy as np
import pandas as pd

# Robust import of /content/.../code.py without colliding with the stdlib 'code' module
def import_module_from_path(module_name: str, file_path: str):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot import module from {file_path}")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


PROJECT_DIR = os.path.dirname(os.path.abspath(__file__))
CODE_PATH = os.path.join(PROJECT_DIR, "code.py")
mod = import_module_from_path("project_code", CODE_PATH)


class TestTargetEngineering(unittest.TestCase):
    def test_binarize_basic_mapping(self):
        s = pd.Series(["NO", "<30", ">30", "NO", ">30", "<30"])
        y = mod.binarize_readmitted(s, unknown_policy="error")
        self.assertEqual(list(y.astype(int)), [0, 1, 0, 0, 0, 1])
        self.assertEqual(y.name, "readmitted_30d")
        self.assertTrue(str(y.dtype).lower() in ("int8", "int64", "int32"))

    def test_binarize_handles_whitespace_and_case(self):
        s = pd.Series([" no ", " <30", ">30 ", "No", "<30", " >30"])
        y = mod.binarize_readmitted(s, unknown_policy="error")
        self.assertEqual(list(y.astype(int)), [0, 1, 0, 0, 1, 0])

    def test_binarize_unknown_raises(self):
        s = pd.Series(["NO", "MAYBE", "<30"])
        with self.assertRaises(mod.TargetEngineeringError):
            _ = mod.binarize_readmitted(s, unknown_policy="error")

    def test_binarize_unknown_to_nan(self):
        s = pd.Series(["NO", "MAYBE", "<30", None])
        y = mod.binarize_readmitted(s, unknown_policy="nan")
        # Expect: NO->0, MAYBE->NA, <30->1, None->NA
        self.assertEqual(int(y.iloc[0]), 0)
        self.assertTrue(pd.isna(y.iloc[1]))
        self.assertEqual(int(y.iloc[2]), 1)
        self.assertTrue(pd.isna(y.iloc[3]))
        self.assertEqual(str(y.dtype), "Int8")

    def test_engineer_targets_adds_columns(self):
        df = pd.DataFrame(
            {
                "readmitted": ["NO", "<30", ">30"],
                "age": ["[50-60)", "[60-70)", "[40-50)"],
            }
        )
        out = mod.engineer_targets(df, add_multiclass=True, drop_source=False, unknown_policy="error")
        self.assertIn("readmitted_30d", out.columns)
        self.assertIn("readmitted_3class", out.columns)
        self.assertIn("readmitted", out.columns)
        self.assertEqual(list(out["readmitted_30d"].astype(int)), [0, 1, 0])
        # multiclass should be categorical with expected categories
        self.assertTrue(pd.api.types.is_categorical_dtype(out["readmitted_3class"]))

    def test_confusion_terms_binary(self):
        y_true = [1, 1, 0, 0, 1, 0]
        y_pred = [1, 0, 0, 1, 1, 0]
        terms = mod.confusion_terms_binary(y_true, y_pred)
        # Manually:
        # TP: positions 0 and 4 => 2
        # FN: position 1 => 1
        # FP: position 3 => 1
        # TN: positions 2 and 5 => 2
        self.assertEqual((terms.tp, terms.fn, terms.fp, terms.tn), (2, 1, 1, 2))

    def test_metrics_multiclass_with_auc(self):
        y_true = ["NO", "<30", ">30", "NO", "<30", ">30"]
        y_pred = ["NO", "<30", "NO", "NO", "<30", ">30"]
        # Probabilities aligned with labels order ("NO", "<30", ">30")
        y_proba = np.array(
            [
                [0.8, 0.1, 0.1],
                [0.1, 0.8, 0.1],
                [0.4, 0.2, 0.4],
                [0.7, 0.2, 0.1],
                [0.1, 0.7, 0.2],
                [0.2, 0.1, 0.7],
            ],
            dtype=float,
        )
        res = mod.metrics_multiclass(y_true, y_pred, y_proba=y_proba)
        for key in ["macro_f1", "weighted_f1", "balanced_accuracy", "ovr_auc_macro"]:
            self.assertIn(key, res)
            self.assertIsInstance(res[key], float)

    def test_keep_multiclass_error_policy(self):
        s = pd.Series(["NO", "<30", "???"])
        with self.assertRaises(mod.TargetEngineeringError):
            _ = mod.keep_multiclass_readmitted(s, unknown_policy="error")

    def test_keep_multiclass_nan_policy(self):
        s = pd.Series(["NO", "<30", "???"])
        out = mod.keep_multiclass_readmitted(s, unknown_policy="nan")
        self.assertTrue(pd.isna(out.iloc[2]))


if __name__ == "__main__":
    unittest.main(verbosity=2)


Overwriting test_target_engineering.py


In [7]:
!python -m unittest -v test_target_engineering.py


test_binarize_basic_mapping (test_target_engineering.TestTargetEngineering.test_binarize_basic_mapping) ... ok
test_binarize_handles_whitespace_and_case (test_target_engineering.TestTargetEngineering.test_binarize_handles_whitespace_and_case) ... ok
test_binarize_unknown_raises (test_target_engineering.TestTargetEngineering.test_binarize_unknown_raises) ... ok
test_binarize_unknown_to_nan (test_target_engineering.TestTargetEngineering.test_binarize_unknown_to_nan) ... ok
test_confusion_terms_binary (test_target_engineering.TestTargetEngineering.test_confusion_terms_binary) ... ok
test_engineer_targets_adds_columns (test_target_engineering.TestTargetEngineering.test_engineer_targets_adds_columns) ... ok
test_keep_multiclass_error_policy (test_target_engineering.TestTargetEngineering.test_keep_multiclass_error_policy) ... ok
test_keep_multiclass_nan_policy (test_target_engineering.TestTargetEngineering.test_keep_multiclass_nan_policy) ... ok
test_metrics_multiclass_with_auc (test_target_

In [2]:
%%writefile section3_dataset_understanding.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd


DEFAULT_MISSING_TOKENS: Tuple[str, ...] = (
    "", "?", "NA", "N/A", "NONE", "NULL", "NAN", "UNKNOWN", "UNKNOWN/INVALID"
)

DEFAULT_ID_COLUMNS: Tuple[str, ...] = (
    "id",
    "url",
    "encounter_id",
    "patient_nbr",
    "patient_id",
    "paper_id",
)

DEFAULT_TEXT_COLUMNS: Tuple[str, ...] = (
    "title",
    "abstract",
    "text",
)

DEFAULT_INTERVAL_COLUMNS: Tuple[str, ...] = (
    "year",
)


def _is_missing_value(x: Any, *, treat_empty_as_missing: bool = True) -> bool:
    if x is None:
        return True
    try:
        if pd.isna(x):
            return True
    except Exception:
        pass
    if treat_empty_as_missing and isinstance(x, str) and x.strip() == "":
        return True
    return False


def _normalize_token(s: str) -> str:
    return s.strip().upper()


def _count_missing_like_tokens(
    series: pd.Series,
    *,
    missing_tokens: Sequence[str] = DEFAULT_MISSING_TOKENS,
) -> Dict[str, int]:
    """
    Count occurrences of missing/unknown encodings in an object/string-like column.
    Counting is case-insensitive and strips whitespace.
    """
    if not (pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype)):
        return {}

    toks = {_normalize_token(t) for t in missing_tokens}
    counts: Dict[str, int] = {t: 0 for t in toks}

    for v in series.astype("object").tolist():
        if v is None:
            continue
        try:
            if pd.isna(v):
                continue
        except Exception:
            pass

        if isinstance(v, str):
            key = _normalize_token(v)
        else:
            key = _normalize_token(str(v))

        if key in counts:
            counts[key] += 1

    # Keep only tokens that actually appear (>0)
    return {k: v for k, v in counts.items() if v > 0}


def _unique_ratio(series: pd.Series) -> float:
    n = len(series)
    if n == 0:
        return 0.0
    return float(series.nunique(dropna=True)) / float(n)


def _avg_text_length(series: pd.Series) -> float:
    vals = []
    for v in series.tolist():
        if _is_missing_value(v, treat_empty_as_missing=True):
            continue
        vals.append(len(str(v)))
    if not vals:
        return 0.0
    return float(np.mean(vals))


def infer_kind_and_scale(
    series: pd.Series,
    col: str,
    *,
    text_columns: Sequence[str] = DEFAULT_TEXT_COLUMNS,
    interval_columns: Sequence[str] = DEFAULT_INTERVAL_COLUMNS,
    ordinal_columns: Sequence[str] = (),
) -> Tuple[str, str]:
    """
    Returns (kind, scale) where:
      - kind  in {"numeric","categorical","text"}
      - scale in {"nominal","ordinal","interval","ratio","text"}
    """
    col_low = col.strip().lower()

    # Forced text columns by name (common in this project)
    if col_low in {c.lower() for c in text_columns}:
        return "text", "text"

    # Numeric
    if pd.api.types.is_numeric_dtype(series.dtype) and not pd.api.types.is_bool_dtype(series.dtype):
        if col_low in {c.lower() for c in interval_columns}:
            return "numeric", "interval"
        return "numeric", "ratio"

    # Bool
    if pd.api.types.is_bool_dtype(series.dtype):
        return "categorical", "nominal"

    # Otherwise object/string => decide text vs categorical by heuristic
    avg_len = _avg_text_length(series)
    if avg_len >= 30.0:
        return "text", "text"

    # Ordinal hint by name override
    if col_low in {c.lower() for c in ordinal_columns}:
        return "categorical", "ordinal"

    return "categorical", "nominal"


def infer_role(
    df: pd.DataFrame,
    col: str,
    kind: str,
    *,
    id_columns: Sequence[str] = DEFAULT_ID_COLUMNS,
    derived_text_column: str = "text",
    id_like_unique_ratio_threshold: float = 0.98,
) -> str:
    """
    Returns a role label to help the report/checklist:
      - "identifier"
      - "raw_text_feature"
      - "derived_feature"
      - "metadata_feature"
      - "numeric_feature"
      - "categorical_feature"
    """
    col_low = col.strip().lower()
    s = df[col]

    if col_low in {c.lower() for c in id_columns}:
        return "identifier"

    # Name-based derived text (common: text = title + abstract)
    if col_low == derived_text_column.lower():
        return "derived_feature"

    # ID-like heuristic (very high uniqueness ratio)
    if _unique_ratio(s) >= id_like_unique_ratio_threshold and kind != "numeric":
        return "identifier"

    # Heuristic: metadata-like
    if col_low in {"year", "venue"}:
        return "metadata_feature"

    if kind == "text":
        return "raw_text_feature"
    if kind == "numeric":
        return "numeric_feature"
    return "categorical_feature"


@dataclass(frozen=True)
class DatasetUnderstandingReport:
    data_dictionary: pd.DataFrame
    missing_summary: pd.DataFrame
    missing_tokens_found: pd.DataFrame
    duplicates_summary: pd.DataFrame
    leakage_risks: pd.DataFrame
    quality_risks: pd.DataFrame


def build_data_dictionary(
    df: pd.DataFrame,
    *,
    text_columns: Sequence[str] = DEFAULT_TEXT_COLUMNS,
    interval_columns: Sequence[str] = DEFAULT_INTERVAL_COLUMNS,
    ordinal_columns: Sequence[str] = (),
    id_columns: Sequence[str] = DEFAULT_ID_COLUMNS,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    n = len(df)

    for col in df.columns:
        s = df[col]
        kind, scale = infer_kind_and_scale(
            s,
            col,
            text_columns=text_columns,
            interval_columns=interval_columns,
            ordinal_columns=ordinal_columns,
        )
        role = infer_role(df, col, kind, id_columns=id_columns)

        miss = int(s.isna().sum())
        empty = 0
        if pd.api.types.is_object_dtype(s.dtype) or pd.api.types.is_string_dtype(s.dtype):
            empty = int((s.astype("object").map(lambda x: isinstance(x, str) and x.strip() == "")).sum())

        nun = int(s.nunique(dropna=True))
        ur = float(nun) / float(n) if n else 0.0

        # Example values (up to 5) excluding missing/empty
        examples = []
        for v in s.tolist():
            if _is_missing_value(v, treat_empty_as_missing=True):
                continue
            examples.append(v)
            if len(examples) >= 5:
                break

        rows.append(
            {
                "column": col,
                "pandas_dtype": str(s.dtype),
                "kind": kind,               # numeric / categorical / text
                "scale": scale,             # nominal / ordinal / interval / ratio / text
                "role": role,               # identifier / feature etc
                "n_rows": n,
                "n_unique": nun,
                "unique_ratio": ur,
                "missing_count": miss,
                "missing_rate": (miss / n) if n else 0.0,
                "empty_string_count": empty,
                "examples": examples,
            }
        )

    return pd.DataFrame(rows).sort_values(["role", "kind", "column"]).reset_index(drop=True)


def summarize_missingness(
    df: pd.DataFrame,
    *,
    treat_empty_as_missing: bool = True,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    n = len(df)

    for col in df.columns:
        s = df[col]
        na = int(s.isna().sum())
        empty = 0
        if treat_empty_as_missing and (pd.api.types.is_object_dtype(s.dtype) or pd.api.types.is_string_dtype(s.dtype)):
            empty = int((s.astype("object").map(lambda x: isinstance(x, str) and x.strip() == "")).sum())

        miss_total = na + empty
        rows.append(
            {
                "column": col,
                "na_count": na,
                "empty_string_count": empty,
                "missing_total": miss_total,
                "missing_rate": (miss_total / n) if n else 0.0,
            }
        )

    return pd.DataFrame(rows).sort_values("missing_rate", ascending=False).reset_index(drop=True)


def detect_missing_tokens(
    df: pd.DataFrame,
    *,
    missing_tokens: Sequence[str] = DEFAULT_MISSING_TOKENS,
) -> pd.DataFrame:
    """
    Detects tokens like '?', 'N/A', 'Unknown', etc., per column (string/object columns).
    Returns a long-form DataFrame: column, token, count
    """
    rows: List[Dict[str, Any]] = []
    for col in df.columns:
        series = df[col]
        counts = _count_missing_like_tokens(series, missing_tokens=missing_tokens)
        for token, cnt in sorted(counts.items()):
            rows.append({"column": col, "token": token, "count": int(cnt)})
    return pd.DataFrame(rows)


def duplicates_report(
    df: pd.DataFrame,
    *,
    key_columns: Sequence[str] = ("url",),
    content_columns: Sequence[str] = ("title", "abstract"),
) -> pd.DataFrame:
    """
    Returns a compact table of duplicate signals:
      - duplicates by key_columns (e.g., URL duplicates)
      - duplicates by content_columns (title+abstract duplicates)
    """
    rows: List[Dict[str, Any]] = []

    def _dup_stats(subset: Sequence[str], name: str) -> None:
        present = [c for c in subset if c in df.columns]
        if not present:
            rows.append(
                {
                    "scope": name,
                    "subset": list(subset),
                    "available_subset": [],
                    "duplicate_rows": 0,
                    "duplicate_groups": 0,
                }
            )
            return

        dup_mask = df.duplicated(subset=present, keep=False)
        duplicate_rows = int(dup_mask.sum())
        # number of duplicated keys/groups (excluding non-duplicated)
        duplicate_groups = int(df.loc[dup_mask, present].drop_duplicates().shape[0])

        rows.append(
            {
                "scope": name,
                "subset": list(subset),
                "available_subset": present,
                "duplicate_rows": duplicate_rows,
                "duplicate_groups": duplicate_groups,
            }
        )

    _dup_stats(key_columns, "key_duplicates")
    _dup_stats(content_columns, "content_duplicates")

    return pd.DataFrame(rows)


def leakage_risk_report(
    df: pd.DataFrame,
    *,
    id_columns: Sequence[str] = DEFAULT_ID_COLUMNS,
    id_like_unique_ratio_threshold: float = 0.98,
) -> pd.DataFrame:
    """
    Flags columns that are likely to cause leakage / trivial memorization in supervised setups
    (or trivial retrieval shortcuts in IR): identifier-like columns, near-unique columns, etc.
    """
    risks: List[Dict[str, Any]] = []
    n = len(df)

    for col in df.columns:
        s = df[col]
        col_low = col.strip().lower()
        ur = _unique_ratio(s)
        nun = int(s.nunique(dropna=True))

        # 1) Name-based ID columns
        if col_low in {c.lower() for c in id_columns}:
            risks.append(
                {
                    "column": col,
                    "risk_type": "identifier_like",
                    "severity": "high",
                    "reason": "Column name matches a known identifier field (e.g., 'url'/'id').",
                    "unique_ratio": ur,
                    "n_unique": nun,
                    "n_rows": n,
                }
            )
            continue

        # 2) ID-like by uniqueness ratio (very close to 1)
        if ur >= id_like_unique_ratio_threshold and not pd.api.types.is_numeric_dtype(s.dtype):
            risks.append(
                {
                    "column": col,
                    "risk_type": "identifier_like",
                    "severity": "medium",
                    "reason": f"Very high uniqueness ratio (>= {id_like_unique_ratio_threshold}).",
                    "unique_ratio": ur,
                    "n_unique": nun,
                    "n_rows": n,
                }
            )

    return pd.DataFrame(risks).sort_values(["severity", "unique_ratio"], ascending=[True, False]).reset_index(drop=True)


def quality_risk_report(
    df: pd.DataFrame,
    *,
    missing_rate_warn: float = 0.20,
    high_cardinality_warn: float = 0.50,
    treat_empty_as_missing: bool = True,
) -> pd.DataFrame:
    """
    Heuristic quality risks: high missingness, high cardinality, constant columns.
    """
    risks: List[Dict[str, Any]] = []
    n = len(df)

    for col in df.columns:
        s = df[col]

        # Missingness risk
        miss_rate = float(
            (_is_missing_value_count(s, treat_empty_as_missing=treat_empty_as_missing) / n) if n else 0.0
        )
        if miss_rate >= missing_rate_warn:
            risks.append(
                {
                    "column": col,
                    "risk_type": "high_missingness",
                    "severity": "medium",
                    "metric": "missing_rate",
                    "value": miss_rate,
                    "threshold": missing_rate_warn,
                    "note": "Consider explicit handling in preprocessing (drop/impute/tokenize missing).",
                }
            )

        # Cardinality risk (mostly for categorical/text columns)
        ur = float(s.nunique(dropna=True) / n) if n else 0.0
        if ur >= high_cardinality_warn and not pd.api.types.is_numeric_dtype(s.dtype):
            risks.append(
                {
                    "column": col,
                    "risk_type": "high_cardinality",
                    "severity": "low",
                    "metric": "unique_ratio",
                    "value": ur,
                    "threshold": high_cardinality_warn,
                    "note": "May lead to sparse features / overfitting if one-hot encoded.",
                }
            )

        # Constant column risk
        nun = int(s.nunique(dropna=True))
        if nun <= 1:
            risks.append(
                {
                    "column": col,
                    "risk_type": "constant_or_almost_constant",
                    "severity": "low",
                    "metric": "n_unique",
                    "value": nun,
                    "threshold": 1,
                    "note": "Usually safe to drop (no predictive signal).",
                }
            )

    return pd.DataFrame(risks).sort_values(["severity", "risk_type", "column"]).reset_index(drop=True)


def _is_missing_value_count(series: pd.Series, *, treat_empty_as_missing: bool) -> int:
    na = int(series.isna().sum())
    if not treat_empty_as_missing:
        return na
    if pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype):
        empty = int((series.astype("object").map(lambda x: isinstance(x, str) and x.strip() == "")).sum())
        return na + empty
    return na


def profile_dataset(
    df_or_records: Union[pd.DataFrame, Sequence[Dict[str, Any]]],
    *,
    expected_keys: Sequence[str] = ("title", "abstract", "url", "venue", "year"),
    text_columns: Sequence[str] = DEFAULT_TEXT_COLUMNS,
    interval_columns: Sequence[str] = DEFAULT_INTERVAL_COLUMNS,
    ordinal_columns: Sequence[str] = (),
    id_columns: Sequence[str] = DEFAULT_ID_COLUMNS,
    missing_tokens: Sequence[str] = DEFAULT_MISSING_TOKENS,
) -> DatasetUnderstandingReport:
    """
    End-to-end helper for Section 3:
      - Builds data dictionary (types/scales/roles)
      - Summarizes missingness + detects missing tokens
      - Detects duplicates
      - Flags leakage risks
      - Flags quality risks

    Accepts either a DataFrame or the raw list[dict] records.
    """
    if isinstance(df_or_records, pd.DataFrame):
        df = df_or_records.copy()
    else:
        # records -> dataframe
        if not isinstance(df_or_records, (list, tuple)):
            raise TypeError(f"df_or_records must be a DataFrame or list/tuple of dicts, got {type(df_or_records)}")
        rows = []
        for i, rec in enumerate(df_or_records):
            if not isinstance(rec, dict):
                raise TypeError(f"Each record must be a dict. Found {type(rec)} at index {i}.")
            rows.append({k: rec.get(k, None) for k in expected_keys})
        df = pd.DataFrame(rows)

    data_dict = build_data_dictionary(
        df,
        text_columns=text_columns,
        interval_columns=interval_columns,
        ordinal_columns=ordinal_columns,
        id_columns=id_columns,
    )
    missing_summary = summarize_missingness(df, treat_empty_as_missing=True)
    missing_tokens_found = detect_missing_tokens(df, missing_tokens=missing_tokens)
    duplicates_summary = duplicates_report(df)
    leakage_risks = leakage_risk_report(df, id_columns=id_columns)
    quality_risks = quality_risk_report(df)

    return DatasetUnderstandingReport(
        data_dictionary=data_dict,
        missing_summary=missing_summary,
        missing_tokens_found=missing_tokens_found,
        duplicates_summary=duplicates_summary,
        leakage_risks=leakage_risks,
        quality_risks=quality_risks,
    )


Writing section3_dataset_understanding.py


In [3]:
%%writefile test_section3_dataset_understanding.py

import unittest
import pandas as pd

import section3_dataset_understanding as s3


class TestSection3DatasetUnderstanding(unittest.TestCase):
    def _tiny_df(self):
        df = pd.DataFrame(
            {
                "title": ["Paper A", "Paper B", "Paper B"],
                "abstract": ["Text A", "Text B", "Text B"],
                "url": ["u1", "u2", "u2"],   # duplicate to test duplicate detection
                "venue": ["EMNLP", "EMNLP", "?"],  # token-like missing/unknown
                "year": [2016, 2017, 2017],
            }
        )
        df["text"] = df["title"] + " " + df["abstract"]  # derived
        return df

    def test_build_data_dictionary_infers_expected_roles_and_scales(self):
        df = self._tiny_df()
        dd = s3.build_data_dictionary(df)

        cols = set(dd["column"].tolist())
        self.assertTrue({"title", "abstract", "url", "venue", "year", "text"}.issubset(cols))

        year_row = dd.loc[dd["column"] == "year"].iloc[0]
        self.assertEqual(year_row["kind"], "numeric")
        self.assertEqual(year_row["scale"], "interval")  # year should be interval by default
        self.assertIn(year_row["role"], {"metadata_feature", "numeric_feature"})

        url_row = dd.loc[dd["column"] == "url"].iloc[0]
        self.assertEqual(url_row["role"], "identifier")

        text_row = dd.loc[dd["column"] == "text"].iloc[0]
        self.assertEqual(text_row["kind"], "text")
        self.assertEqual(text_row["role"], "derived_feature")

        title_row = dd.loc[dd["column"] == "title"].iloc[0]
        self.assertEqual(title_row["kind"], "text")
        self.assertEqual(title_row["role"], "raw_text_feature")

    def test_detect_missing_tokens_finds_question_mark(self):
        df = self._tiny_df()
        mt = s3.detect_missing_tokens(df, missing_tokens=("?", "N/A"))

        # We expect venue has "?" exactly once
        found = mt[(mt["column"] == "venue") & (mt["token"] == "?")]
        self.assertEqual(int(found["count"].iloc[0]), 1)

    def test_missing_summary_counts_empty_strings(self):
        df = self._tiny_df()
        df.loc[1, "abstract"] = "   "  # empty after strip
        ms = s3.summarize_missingness(df, treat_empty_as_missing=True)

        abs_row = ms.loc[ms["column"] == "abstract"].iloc[0]
        self.assertGreaterEqual(int(abs_row["empty_string_count"]), 1)
        self.assertGreaterEqual(int(abs_row["missing_total"]), 1)

    def test_duplicates_report_detects_key_duplicates(self):
        df = self._tiny_df()
        rep = s3.duplicates_report(df, key_columns=("url",), content_columns=("title", "abstract"))

        key_row = rep.loc[rep["scope"] == "key_duplicates"].iloc[0]
        self.assertEqual(key_row["available_subset"], ["url"])
        self.assertGreaterEqual(int(key_row["duplicate_rows"]), 2)  # u2 repeated => two rows duplicated

    def test_leakage_risk_report_flags_url_as_identifier_like(self):
        df = self._tiny_df()
        lr = s3.leakage_risk_report(df, id_columns=("url", "id"))

        self.assertTrue("column" in lr.columns)
        self.assertTrue(any(lr["column"] == "url"))

    def test_profile_dataset_accepts_records_list(self):
        records = [
            {"title": "T1", "abstract": "A1", "url": "u1", "venue": "EMNLP", "year": 2016},
            {"title": "T2", "abstract": "A2", "url": "u2", "venue": "EMNLP", "year": 2017},
        ]
        report = s3.profile_dataset(records)
        self.assertTrue(hasattr(report, "data_dictionary"))
        self.assertTrue(hasattr(report, "missing_summary"))
        self.assertTrue(hasattr(report, "duplicates_summary"))


if __name__ == "__main__":
    unittest.main(verbosity=2)


Writing test_section3_dataset_understanding.py


In [4]:
!python -m unittest -v test_section3_dataset_understanding.py


test_build_data_dictionary_infers_expected_roles_and_scales (test_section3_dataset_understanding.TestSection3DatasetUnderstanding.test_build_data_dictionary_infers_expected_roles_and_scales) ... ok
test_detect_missing_tokens_finds_question_mark (test_section3_dataset_understanding.TestSection3DatasetUnderstanding.test_detect_missing_tokens_finds_question_mark) ... ok
test_duplicates_report_detects_key_duplicates (test_section3_dataset_understanding.TestSection3DatasetUnderstanding.test_duplicates_report_detects_key_duplicates) ... ok
test_leakage_risk_report_flags_url_as_identifier_like (test_section3_dataset_understanding.TestSection3DatasetUnderstanding.test_leakage_risk_report_flags_url_as_identifier_like) ... ok
test_missing_summary_counts_empty_strings (test_section3_dataset_understanding.TestSection3DatasetUnderstanding.test_missing_summary_counts_empty_strings) ... ok
test_profile_dataset_accepts_records_list (test_section3_dataset_understanding.TestSection3DatasetUnderstanding.